# LeetCode 3187 — Peaks in Array bằng Python

- Link: https://leetcode.com/problems/peaks-in-array/
- Độ khó: **Hard**
- Chủ đề: **Fenwick tree (BIT) / Segment tree** trên mảng chỉ báo `peak[i] ∈ {0, 1}`; truy vấn đếm đỉnh trong **phần trong** `(l, r)`; cập nhật một phần tử chỉ ảnh hưởng `peak` tại `i − 1, i, i + 1`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countOfPeaks(self, nums: List[int], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Quy bài về mảng chỉ báo `peak[i] = 1` nếu `nums[i−1] < nums[i] > nums[i+1]`.
2. Đỉnh của **mảng con** `nums[l..r]` là các `i` với `l < i < r` và `peak[i] = 1` — vì đầu/cuối mảng con không được tính, và với `l < i < r` hai hàng xóm đều nằm trong mảng con.
3. Truy vấn tổng đoạn + cập nhật điểm ⇒ **Fenwick tree** `O(log n)`.
4. Cập nhật `nums[i]` chỉ làm đổi `peak[i−1], peak[i], peak[i+1]` ⇒ tính lại 3 chỉ báo.
5. Cẩn thận đoạn rỗng khi `r − l < 2`.

## 2. Đề bài & ví dụ

**Đỉnh** của mảng là phần tử lớn hơn cả phần tử trước và sau nó (phần tử đầu và cuối không bao giờ là đỉnh). Cho `nums` và các truy vấn:

- `[1, l, r]`: đếm số đỉnh trong mảng con `nums[l..r]`;
- `[2, idx, val]`: gán `nums[idx] = val`.

Trả về danh sách kết quả các truy vấn loại 1 theo thứ tự.

**Ví dụ 1:** `nums = [3,1,4,2,5]`, `queries = [[2,3,4],[1,0,4]]` → `[0]`.

**Ví dụ 2:** `nums = [4,1,4,2,1,5]`, `queries = [[2,2,4],[1,0,2],[1,0,4]]` → `[0,1]`.

**Ràng buộc:** `3 ≤ n ≤ 10^5`; `1 ≤ nums[i], val ≤ 10^5`; `1 ≤ queries.length ≤ 10^5`.

In [1]:
from typing import List
import random, time

examples = [([3, 1, 4, 2, 5], [[2, 3, 4], [1, 0, 4]], [0]),
            ([4, 1, 4, 2, 1, 5], [[2, 2, 4], [1, 0, 2], [1, 0, 4]], [0, 1])]
for a, q, exp in examples:
    print(a, q, '-> kỳ vọng', exp)

[3, 1, 4, 2, 5] [[2, 3, 4], [1, 0, 4]] -> kỳ vọng [0]
[4, 1, 4, 2, 1, 5] [[2, 2, 4], [1, 0, 2], [1, 0, 4]] -> kỳ vọng [0, 1]


## 3. Kiến thức nền

### 3.1. Fenwick tree (Binary Indexed Tree)

Lưu `tree[i]` = tổng của đoạn `(i − lowbit(i), i]` (chỉ số 1). `add(i, d)`: đi lên `i += i & −i`; `prefix(i)`: đi xuống `i −= i & −i`. Cả hai `O(log n)`. Xây trong `O(n)` bằng cách "đẩy" mỗi nút lên cha một lần.

In [2]:
class BIT:
    def __init__(self, arr):                      # arr chỉ số 0
        n = len(arr); self.n = n; t = [0] + list(arr)
        for i in range(1, n + 1):                 # dựng O(n)
            j = i + (i & -i)
            if j <= n:
                t[j] += t[i]
        self.t = t
    def add(self, i, d):                          # i chỉ số 0
        i += 1
        while i <= self.n:
            self.t[i] += d; i += i & -i
    def prefix(self, i):                          # tổng arr[0..i-1]
        s = 0
        while i > 0:
            s += self.t[i]; i -= i & -i
        return s
    def range_sum(self, l, r):                    # tổng arr[l..r], rỗng nếu l > r
        return 0 if l > r else self.prefix(r + 1) - self.prefix(l)

b = BIT([1, 0, 1, 1, 0, 1])
print('tổng [1..4] =', b.range_sum(1, 4)); b.add(1, 1); print('sau khi +1 tại 1:', b.range_sum(1, 4))

tổng [1..4] = 2
sau khi +1 tại 1: 3


### 3.2. Vì sao chỉ cần `(l, r)` và chỉ 3 vị trí khi cập nhật

- Với `l < i < r`, `nums[i−1]` và `nums[i+1]` đều thuộc mảng con ⇒ "đỉnh trong mảng con" ⇔ `peak[i] = 1` (đỉnh trong mảng gốc). `i = l` và `i = r` không bao giờ tính.
- `peak[j]` chỉ đọc `nums[j−1], nums[j], nums[j+1]` ⇒ đổi `nums[idx]` chỉ ảnh hưởng `j ∈ {idx−1, idx, idx+1}`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Không cần biết đỉnh là "giá trị bao nhiêu" — chỉ cần một mảng 0/1 và đếm số 1 trong đoạn. Mỗi lần cập nhật như "thay 3 bóng đèn" quanh `idx`. Ví dụ 2: `peak = [0, 0, 1, 0, 0, 0]`; truy vấn `[1, 0, 2]` xét `i ∈ (0, 2) = {1}` ⇒ `0`; truy vấn `[1, 0, 4]` xét `{1, 2, 3}` ⇒ `1`.

## 5. Cách 1 — Brute Force: mô phỏng trực tiếp

In [3]:
def brute(nums, queries):
    a = list(nums); res = []
    for typ, x, y in queries:
        if typ == 1:
            res.append(sum(1 for i in range(x + 1, y) if a[i - 1] < a[i] > a[i + 1]))
        else:
            a[x] = y
    return res

for a, q, exp in examples:
    print('brute:', brute(a, q), '| kỳ vọng', exp)

brute: [0] | kỳ vọng [0]
brute: [0, 1] | kỳ vọng [0, 1]


**Độ phức tạp:** `O(n)` mỗi truy vấn loại 1 ⇒ `O(n · q)` = `10^{10}` xấu nhất.

## 6. Ý tưởng thuật toán / bước đột phá

```
peak[i] = 1 nếu 0 < i < n-1 và nums[i-1] < nums[i] > nums[i+1]
BIT trên peak
[1, l, r]:  ans = BIT.sum(l+1, r-1)   (0 nếu r - l < 2)
[2, i, v]:  với j ∈ {i-1, i, i+1} ∩ [1, n-2]: bỏ peak[j] cũ
            nums[i] = v
            với j đó: tính lại peak[j], cộng vào BIT
```

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Fenwick tree (lời giải chính)

In [4]:
def fenwick_solution(nums, queries):
    a = list(nums); n = len(a)
    def is_peak(i):
        return 1 if 0 < i < n - 1 and a[i - 1] < a[i] > a[i + 1] else 0
    pk = [is_peak(i) for i in range(n)]
    bit = BIT(pk); res = []
    for typ, x, y in queries:
        if typ == 1:
            res.append(bit.range_sum(x + 1, y - 1))
        else:
            a[x] = y
            for j in (x - 1, x, x + 1):
                if 0 <= j < n:
                    v = is_peak(j)
                    if v != pk[j]:
                        bit.add(j, v - pk[j]); pk[j] = v
    return res

for a, q, exp in examples:
    print('Fenwick:', fenwick_solution(a, q), '| kỳ vọng', exp)

Fenwick: [0] | kỳ vọng [0]
Fenwick: [0, 1] | kỳ vọng [0, 1]


### 7.2. Cách 3 — Segment tree lặp (bottom-up)

In [5]:
def segtree_solution(nums, queries):
    a = list(nums); n = len(a)
    def is_peak(i):
        return 1 if 0 < i < n - 1 and a[i - 1] < a[i] > a[i + 1] else 0
    size = 1
    while size < n:
        size *= 2
    seg = [0] * (2 * size)
    for i in range(n):
        seg[size + i] = is_peak(i)
    for i in range(size - 1, 0, -1):
        seg[i] = seg[2 * i] + seg[2 * i + 1]
    def setv(i, v):
        i += size; seg[i] = v; i //= 2
        while i:
            seg[i] = seg[2 * i] + seg[2 * i + 1]; i //= 2
    def query(l, r):                               # [l, r]
        s = 0; l += size; r += size + 1
        while l < r:
            if l & 1: s += seg[l]; l += 1
            if r & 1: r -= 1; s += seg[r]
            l //= 2; r //= 2
        return s
    res = []
    for typ, x, y in queries:
        if typ == 1:
            res.append(query(x + 1, y - 1) if y - x >= 2 else 0)
        else:
            a[x] = y
            for j in (x - 1, x, x + 1):
                if 0 <= j < n:
                    setv(j, is_peak(j))
    return res

for a, q, exp in examples:
    print('segment tree:', segtree_solution(a, q), '| kỳ vọng', exp)

segment tree: [0] | kỳ vọng [0]
segment tree: [0, 1] | kỳ vọng [0, 1]


### 7.3. Cách 4 — Chia căn (sqrt decomposition)

Chia `peak` thành khối cỡ `B ≈ √n`, lưu tổng mỗi khối. Truy vấn: cộng các khối trọn + phần lẻ hai đầu ⇒ `O(√n)`; cập nhật `O(1)`.

In [6]:
def sqrt_solution(nums, queries):
    a = list(nums); n = len(a); B = max(1, int(n ** 0.5))
    def is_peak(i):
        return 1 if 0 < i < n - 1 and a[i - 1] < a[i] > a[i + 1] else 0
    pk = [is_peak(i) for i in range(n)]
    blk = [0] * (n // B + 1)
    for i, v in enumerate(pk):
        blk[i // B] += v
    res = []
    for typ, x, y in queries:
        if typ == 1:
            l, r = x + 1, y - 1; s = 0
            if l <= r:
                bl, br = l // B, r // B
                if bl == br:
                    s = sum(pk[l:r + 1])
                else:
                    s = sum(pk[l:(bl + 1) * B]) + sum(blk[bl + 1:br]) + sum(pk[br * B:r + 1])
            res.append(s)
        else:
            a[x] = y
            for j in (x - 1, x, x + 1):
                if 0 <= j < n:
                    v = is_peak(j)
                    if v != pk[j]:
                        blk[j // B] += v - pk[j]; pk[j] = v
    return res

for a, q, exp in examples:
    print('sqrt:', sqrt_solution(a, q), '| kỳ vọng', exp)

sqrt: [0] | kỳ vọng [0]
sqrt: [0, 1] | kỳ vọng [0, 1]


## 8. TRACE từng bước

Trace cách 2: in mảng `nums`, `peak` và kết quả sau mỗi truy vấn.

In [7]:
def trace(nums, queries):
    a = list(nums); n = len(a)
    is_peak = lambda i: 1 if 0 < i < n - 1 and a[i - 1] < a[i] > a[i + 1] else 0
    pk = [is_peak(i) for i in range(n)]; bit = BIT(pk)
    print('ban đầu: nums =', a, '| peak =', pk)
    for typ, x, y in queries:
        if typ == 1:
            ans = bit.range_sum(x + 1, y - 1)
            print(f'[1, {x}, {y}]: đếm peak[{x + 1}..{y - 1}] = {ans}')
        else:
            a[x] = y; changed = []
            for j in (x - 1, x, x + 1):
                if 0 <= j < n:
                    v = is_peak(j)
                    if v != pk[j]:
                        bit.add(j, v - pk[j]); changed.append((j, pk[j], v)); pk[j] = v
            print(f'[2, {x}, {y}]: nums = {a} | peak = {pk} | đổi (j, cũ, mới): {changed}')

trace([5, 1, 6, 2, 7, 3, 8], [[1, 0, 6], [2, 3, 9], [1, 0, 6], [1, 2, 4], [2, 4, 1], [1, 1, 5], [1, 3, 3]])

ban đầu: nums = [5, 1, 6, 2, 7, 3, 8] | peak = [0, 0, 1, 0, 1, 0, 0]
[1, 0, 6]: đếm peak[1..5] = 2
[2, 3, 9]: nums = [5, 1, 6, 9, 7, 3, 8] | peak = [0, 0, 0, 1, 0, 0, 0] | đổi (j, cũ, mới): [(2, 1, 0), (3, 0, 1), (4, 1, 0)]
[1, 0, 6]: đếm peak[1..5] = 1
[1, 2, 4]: đếm peak[3..3] = 1
[2, 4, 1]: nums = [5, 1, 6, 9, 1, 3, 8] | peak = [0, 0, 0, 1, 0, 0, 0] | đổi (j, cũ, mới): []
[1, 1, 5]: đếm peak[2..4] = 1
[1, 3, 3]: đếm peak[4..2] = 0


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def countOfPeaks(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        a = nums
        n = len(a)
        pk = [0] * n
        for i in range(1, n - 1):
            if a[i - 1] < a[i] > a[i + 1]:
                pk[i] = 1
        t = [0] + pk[:]
        for i in range(1, n + 1):
            j = i + (i & -i)
            if j <= n:
                t[j] += t[i]

        def add(i, d):
            i += 1
            while i <= n:
                t[i] += d
                i += i & -i

        def prefix(i):
            s = 0
            while i > 0:
                s += t[i]
                i -= i & -i
            return s

        res = []
        for typ, x, y in queries:
            if typ == 1:
                res.append(prefix(y) - prefix(x + 1) if y - x >= 2 else 0)   # tổng pk[x+1..y-1]
            else:
                a[x] = y
                for j in (x - 1, x, x + 1):
                    if 0 < j < n - 1:
                        v = 1 if a[j - 1] < a[j] > a[j + 1] else 0
                        if v != pk[j]:
                            add(j, v - pk[j])
                            pk[j] = v
        return res

for a, q, exp in examples:
    got = Solution().countOfPeaks(list(a), q)
    print(got, 'OK' if got == exp else 'SAI')

[0] OK
[0, 1] OK


(Lời giải nộp sửa trực tiếp `nums` — LeetCode cho phép; trong notebook luôn truyền bản sao `list(a)`.)

## 10. Vì sao thuật toán đúng

1. **Bất biến:** sau mỗi truy vấn, `pk[j] = [nums[j−1] < nums[j] > nums[j+1]]` với mọi `0 < j < n−1` và `BIT` lưu đúng tổng tiền tố của `pk`. Cập nhật `nums[x]` chỉ đổi các `pk[j]` có `j ∈ {x−1, x, x+1}` (mục 3.2); ta tính lại đúng những vị trí này.
2. **Truy vấn:** đỉnh của `nums[l..r]` là `{ i : l < i < r, pk[i] = 1 }` ⇒ đáp án `Σ_{i=l+1}^{r−1} pk[i] = prefix(r) − prefix(l+1)`; khi `r − l < 2` tập rỗng ⇒ 0.

## 11. Phân tích Time / Space Complexity

| Cách | Dựng | Truy vấn | Cập nhật | Space |
|---|---|---|---|---|
| 1. Brute | — | `O(n)` | `O(1)` | `O(1)` |
| 2. Fenwick | `O(n)` | `O(log n)` | `O(log n)` | `O(n)` |
| 3. Segment tree | `O(n)` | `O(log n)` | `O(log n)` | `O(n)` |
| 4. Chia căn | `O(n)` | `O(√n)` | `O(1)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'fenwick': fenwick_solution, 'segtree': segtree_solution, 'sqrt': sqrt_solution,
         'submit': lambda a, q: Solution().countOfPeaks(list(a), q)}
edge = [([1, 2, 1], [[1, 0, 2], [1, 0, 1], [1, 1, 1], [2, 1, 1], [1, 0, 2]], [1, 0, 0, 0]),
        ([1, 3, 1, 3, 1], [[1, 0, 4], [1, 1, 3], [2, 2, 5], [1, 0, 4], [1, 1, 3]], [2, 0, 1, 1]),
        ([2, 2, 2], [[1, 0, 2], [2, 1, 3], [1, 0, 2], [2, 0, 5], [1, 0, 2]], [0, 1, 0])]
for a, q, exp in examples + edge:
    assert brute(a, q) == exp, (a, q, brute(a, q))
    for name, f in impls.items():
        assert f(a, q) == exp, (name, a, q)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[1, 3, 1, 3, 1]` truy vấn `[1, 1, 3]` — hai số 3 nằm ở **đầu/cuối** mảng con ⇒ 0. Gán `nums[2] = 5` làm hai đỉnh 3 mất và 5 thành đỉnh duy nhất. `[2, 2, 2]`: bằng nhau không phải đỉnh (cần lớn hơn **ngặt**).

### 12.2. Đối chiếu ngẫu nhiên với brute

In [10]:
rng = random.Random(3187)
def rand_case(n, q, V):
    a = [rng.randint(1, V) for _ in range(n)]; qs = []
    for _ in range(q):
        if rng.random() < 0.5:
            l = rng.randint(0, n - 1); r = rng.randint(l, n - 1); qs.append([1, l, r])
        else:
            qs.append([2, rng.randint(0, n - 1), rng.randint(1, V)])
    return a, qs

cnt = 0
for _ in range(300):
    a, qs = rand_case(rng.randint(3, 12), rng.randint(1, 15), rng.choice([2, 3, 10]))
    exp = brute(a, qs)
    for name, f in impls.items():
        assert f(a, qs) == exp, (name, a, qs)
    cnt += 1
print('Khớp brute trên', cnt, 'bộ ngẫu nhiên.')

Khớp brute trên 300 bộ ngẫu nhiên.


### 12.3. Ca lớn `n = q = 10^5`

In [11]:
N = 10 ** 5
cases = {
    'ngẫu nhiên': rand_case(N, N, 10 ** 5),
    'răng cưa, hỏi toàn mảng': ([1 if i % 2 == 0 else 2 for i in range(N)], [[1, 0, N - 1]] * N),
    'giá trị nhỏ': rand_case(N, N, 3),
}
for name, (a, qs) in cases.items():
    t = time.time(); x = Solution().countOfPeaks(list(a), qs); t1 = time.time() - t
    t = time.time(); y = segtree_solution(a, qs); t2 = time.time() - t
    assert x == y
    print(f'{name:24s}: {len(x):6d} câu trả lời | nộp {t1:.2f}s | segtree {t2:.2f}s')
assert Solution().countOfPeaks([1 if i % 2 == 0 else 2 for i in range(N)], [[1, 0, N - 1]])[0] == (N - 1) // 2
print('Răng cưa 1,2,1,2,…: (N − 1) // 2 đỉnh: đúng.')

ngẫu nhiên              :  50052 câu trả lời | nộp 0.31s | segtree 0.78s


răng cưa, hỏi toàn mảng : 100000 câu trả lời | nộp 0.21s | segtree 0.44s


giá trị nhỏ             :  49843 câu trả lời | nộp 0.28s | segtree 0.76s
Răng cưa 1,2,1,2,…: (N − 1) // 2 đỉnh: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm cả hai đầu mảng con (`peak[l..r]` thay vì `peak[l+1..r−1]`)

In [12]:
def wrong_inclusive(nums, queries):
    a = list(nums); n = len(a); res = []
    for typ, x, y in queries:
        if typ == 1:
            res.append(sum(1 for i in range(x, y + 1) if 0 < i < n - 1 and a[i - 1] < a[i] > a[i + 1]))   # SAI
        else:
            a[x] = y
    return res

args = ([1, 3, 1, 3, 1], [[1, 1, 3]])
print('sai :', wrong_inclusive(*args), '| đúng:', Solution().countOfPeaks(list(args[0]), args[1]))

sai : [2] | đúng: [0]


**Vì sao sai:** `nums[1] = 3` là đỉnh của mảng gốc nhưng là **phần tử đầu** của mảng con `[3, 1, 3]` ⇒ không tính.

### Lỗi 2 — Khi cập nhật chỉ tính lại `peak[idx]` (quên hai hàng xóm)

In [13]:
def wrong_update_self_only(nums, queries):
    a = list(nums); n = len(a)
    is_peak = lambda i: 1 if 0 < i < n - 1 and a[i - 1] < a[i] > a[i + 1] else 0
    pk = [is_peak(i) for i in range(n)]; bit = BIT(pk); res = []
    for typ, x, y in queries:
        if typ == 1:
            res.append(bit.range_sum(x + 1, y - 1))
        else:
            a[x] = y
            v = is_peak(x)                                            # SAI: bỏ x-1, x+1
            if v != pk[x]:
                bit.add(x, v - pk[x]); pk[x] = v
    return res

args = ([1, 3, 1, 1], [[2, 2, 5], [1, 0, 3]])
print('sai :', wrong_update_self_only(*args), '| đúng:', Solution().countOfPeaks(list(args[0]), args[1]))

sai : [2] | đúng: [1]


**Vì sao sai:** tăng `nums[2]` lên 5 làm `nums[1] = 3` **mất** tư cách đỉnh (giờ `3 < 5`) — `peak[1]` phải đổi theo.

### Lỗi 3 — Không xử lý đoạn rỗng: `prefix(r) − prefix(l + 1)` khi `r = l`

In [14]:
def wrong_empty_range(nums, queries):
    a = list(nums); n = len(a)
    is_peak = lambda i: 1 if 0 < i < n - 1 and a[i - 1] < a[i] > a[i + 1] else 0
    bit = BIT([is_peak(i) for i in range(n)]); res = []
    for typ, x, y in queries:
        if typ == 1:
            res.append(bit.prefix(y) - bit.prefix(x + 1))             # SAI khi y - x < 2
    return res

args = ([1, 2, 1], [[1, 1, 1]])
print('sai :', wrong_empty_range(*args), '| đúng:', Solution().countOfPeaks(list(args[0]), args[1]))

sai : [-1] | đúng: [0]


**Vì sao sai:** khi `l = r`, công thức thành `prefix(l) − prefix(l + 1) = −peak[l]` — có thể ra **số âm**.

### Lỗi 4 — Dùng `≥` thay vì `>` (coi "bằng" là đỉnh)

In [15]:
def wrong_non_strict(nums, queries):
    a = list(nums); res = []
    for typ, x, y in queries:
        if typ == 1:
            res.append(sum(1 for i in range(x + 1, y) if a[i - 1] <= a[i] >= a[i + 1]))   # SAI
        else:
            a[x] = y
    return res

print('sai :', wrong_non_strict(*examples[0][:2]), '| đúng:', Solution().countOfPeaks(list(examples[0][0]), examples[0][1]))

sai : [1] | đúng: [0]


**Vì sao sai:** sau cập nhật, `[3, 1, 4, 4, 5]` có hai số 4 liền nhau — không số nào **lớn hơn ngặt** cả hai bên.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm phần tử có tính chất cục bộ trong đoạn + cập nhật điểm" ⇒ mảng chỉ báo + BIT.
- Tính chất phụ thuộc hàng xóm ⇒ cập nhật lan ra **bán kính** của tính chất (ở đây 1).
- Luôn kiểm đoạn rỗng/đoạn ngắn khi dùng hiệu tiền tố.

## 15. Biến thể của bài

1. Đếm **thung lũng** (nhỏ hơn hai bên) ⇒ đổi điều kiện.
2. Đỉnh theo nghĩa "lớn hơn `k` phần tử mỗi bên" ⇒ bán kính cập nhật `k`.
3. Truy vấn "đỉnh thứ `k` trong đoạn" ⇒ tìm kiếm trên BIT (walk).
4. Cập nhật **cộng đoạn** `[l, r] += v` ⇒ `peak` chỉ đổi ở hai biên `l−1, l, r, r+1` (bên trong chênh lệch giữ nguyên).

In [16]:
# Biến thể 4: cập nhật cộng đoạn — chỉ cần tính lại peak quanh hai biên
def range_add_solution(nums, queries):
    a = list(nums); n = len(a)
    is_peak = lambda i: 1 if 0 < i < n - 1 and a[i - 1] < a[i] > a[i + 1] else 0
    pk = [is_peak(i) for i in range(n)]; bit = BIT(pk); res = []
    for q in queries:
        if q[0] == 1:
            res.append(bit.range_sum(q[1] + 1, q[2] - 1))
        else:
            _, l, r, v = q
            for i in range(l, r + 1):                   # (mô phỏng giá trị; cấu trúc thật dùng BIT hiệu)
                a[i] += v
            for j in {l - 1, l, r, r + 1}:
                if 0 <= j < n:
                    nv = is_peak(j)
                    if nv != pk[j]:
                        bit.add(j, nv - pk[j]); pk[j] = nv
    return res

def range_add_brute(nums, queries):
    a = list(nums); res = []
    for q in queries:
        if q[0] == 1:
            res.append(sum(1 for i in range(q[1] + 1, q[2]) if a[i - 1] < a[i] > a[i + 1]))
        else:
            for i in range(q[1], q[2] + 1):
                a[i] += q[3]
    return res

for _ in range(200):
    n = rng.randint(3, 10); a = [rng.randint(1, 4) for _ in range(n)]; qs = []
    for _ in range(rng.randint(1, 10)):
        l = rng.randint(0, n - 1); r = rng.randint(l, n - 1)
        qs.append([1, l, r] if rng.random() < 0.5 else [2, l, r, rng.randint(-2, 2)])
    assert range_add_solution(a, qs) == range_add_brute(a, qs)
print('Biến thể cộng đoạn: chỉ tính lại peak tại l−1, l, r, r+1 — khớp brute.')

Biến thể cộng đoạn: chỉ tính lại peak tại l−1, l, r, r+1 — khớp brute.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Truy vấn / cập nhật | Ghi chú |
|---|---|---|---|
| 1 | Mô phỏng | `O(n)` / `O(1)` | Oracle |
| 2 | Fenwick | `O(log n)` / `O(log n)` | **Nộp bài**, ngắn |
| 3 | Segment tree | `O(log n)` / `O(log n)` | Mở rộng được (min/max…) |
| 4 | Chia căn | `O(√n)` / `O(1)` | Dễ hiểu |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [1, 5, 2, 6, 3]`, `queries = [[1, 0, 4], [1, 1, 3]]`.

**Bài 2 (tính tay):** `nums = [2, 1, 2, 1, 2]`, `queries = [[2, 2, 0], [1, 0, 4], [2, 1, 3], [1, 0, 4]]`.

**Bài 3 (điền TODO):** thêm truy vấn loại 3 `[3, l, r]`: đếm **thung lũng** trong mảng con.

**Bài 4 (tự viết lại):** truy vấn `[1, l, r, k]`: trả về **chỉ số** của đỉnh thứ `k` (đếm từ 1) trong mảng con, hoặc `-1`.

**Bài 5 (mini challenge):** đỉnh "rộng": `nums[i]` lớn hơn **2** phần tử mỗi bên (`i−2, i−1, i+1, i+2`); đầu/cuối 2 phần tử của mảng con không tính.

In [17]:
def peaks_and_valleys_todo(nums, queries):
    # TODO
    raise NotImplementedError

def kth_peak_todo(nums, queries):
    # TODO
    raise NotImplementedError

def wide_peaks_todo(nums, queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `peak = [0, 1, 0, 1, 0]`. (2) `[1, 0, 4]` ⇒ `peak[1..3]`. (3) `[2, 0]`.

**Bài 2.** (1) Sau `nums[2] = 0`: `[2, 1, 0, 1, 2]` — không đỉnh. (2) Sau `nums[1] = 3`: `[2, 3, 0, 1, 2]` — đỉnh tại 1. (3) `[0, 1]`.

**Bài 3.** (1) Thêm mảng chỉ báo `valley` và BIT thứ hai. (2) Cập nhật cả hai tại `x−1, x, x+1`. (3) Truy vấn như đỉnh.

**Bài 4.** (1) `k`-th đỉnh trong đoạn = đỉnh thứ `prefix(l+1) + k` của toàn mảng. (2) Tìm trên BIT bằng "walk" theo lũy thừa 2. (3) Kiểm vị trí `< r`.

**Bài 5.** (1) Điều kiện phụ thuộc bán kính 2. (2) Cập nhật tính lại `x−2..x+2`. (3) Truy vấn đếm trên `[l+2, r−2]`.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().countOfPeaks([1, 5, 2, 6, 3], [[1, 0, 4], [1, 1, 3]]) == [2, 0]
assert Solution().countOfPeaks([2, 1, 2, 1, 2], [[2, 2, 0], [1, 0, 4], [2, 1, 3], [1, 0, 4]]) == [0, 1]

def peaks_and_valleys(nums, queries):
    a = list(nums); n = len(a)
    ip = lambda i: 1 if 0 < i < n - 1 and a[i - 1] < a[i] > a[i + 1] else 0
    iv = lambda i: 1 if 0 < i < n - 1 and a[i - 1] > a[i] < a[i + 1] else 0
    P = [ip(i) for i in range(n)]; V = [iv(i) for i in range(n)]
    bp, bv = BIT(P), BIT(V); res = []
    for typ, x, y in queries:
        if typ == 1:
            res.append(bp.range_sum(x + 1, y - 1))
        elif typ == 3:
            res.append(bv.range_sum(x + 1, y - 1))
        else:
            a[x] = y
            for j in (x - 1, x, x + 1):
                if 0 <= j < n:
                    p, v = ip(j), iv(j)
                    if p != P[j]: bp.add(j, p - P[j]); P[j] = p
                    if v != V[j]: bv.add(j, v - V[j]); V[j] = v
    return res

def brute_pv(nums, queries):
    a = list(nums); res = []
    for typ, x, y in queries:
        if typ == 1:
            res.append(sum(1 for i in range(x + 1, y) if a[i - 1] < a[i] > a[i + 1]))
        elif typ == 3:
            res.append(sum(1 for i in range(x + 1, y) if a[i - 1] > a[i] < a[i + 1]))
        else:
            a[x] = y
    return res

for _ in range(200):
    a, qs = rand_case(rng.randint(3, 10), rng.randint(1, 12), 4)
    qs = [[3, q[1], q[2]] if q[0] == 1 and rng.random() < 0.5 else q for q in qs]
    assert peaks_and_valleys(a, qs) == brute_pv(a, qs)
print('Bài 1: [2, 0] | Bài 2: [0, 1] | Bài 3: đỉnh + thung lũng khớp brute.')

Bài 1: [2, 0] | Bài 2: [0, 1] | Bài 3: đỉnh + thung lũng khớp brute.


In [19]:
# Bài 4, 5
def kth_peak(nums, queries):
    a = list(nums); n = len(a)
    ip = lambda i: 1 if 0 < i < n - 1 and a[i - 1] < a[i] > a[i + 1] else 0
    P = [ip(i) for i in range(n)]; bit = BIT(P); res = []
    LOG = n.bit_length()
    def find_kth(k):                       # vị trí (0-index) của số 1 thứ k trên toàn mảng
        pos = 0
        for p in range(LOG, -1, -1):
            nxt = pos + (1 << p)
            if nxt <= n and bit.t[nxt] < k:
                pos = nxt; k -= bit.t[nxt]
        return pos                         # chỉ số 1-index pos+1 -> 0-index pos
    for q in queries:
        if q[0] == 1:
            _, l, r, k = q
            if r - l < 2:
                res.append(-1); continue
            before = bit.prefix(l + 1); total = bit.range_sum(l + 1, r - 1)
            res.append(find_kth(before + k) if k <= total else -1)
        else:
            _, x, y = q; a[x] = y
            for j in (x - 1, x, x + 1):
                if 0 <= j < n:
                    v = ip(j)
                    if v != P[j]: bit.add(j, v - P[j]); P[j] = v
    return res

def wide_peaks(nums, queries):
    a = list(nums); n = len(a)
    iw = lambda i: 1 if 1 < i < n - 2 and all(a[i] > a[j] for j in (i - 2, i - 1, i + 1, i + 2)) else 0
    W = [iw(i) for i in range(n)]; bit = BIT(W); res = []
    for typ, x, y in queries:
        if typ == 1:
            res.append(bit.range_sum(x + 2, y - 2))
        else:
            a[x] = y
            for j in range(x - 2, x + 3):
                if 0 <= j < n:
                    v = iw(j)
                    if v != W[j]: bit.add(j, v - W[j]); W[j] = v
    return res

for _ in range(200):
    n = rng.randint(3, 12); a = [rng.randint(1, 6) for _ in range(n)]; qs = []; qs2 = []
    for _ in range(rng.randint(1, 12)):
        if rng.random() < 0.5:
            l = rng.randint(0, n - 1); r = rng.randint(l, n - 1)
            qs.append([1, l, r, rng.randint(1, 3)]); qs2.append([1, l, r])
        else:
            q = [2, rng.randint(0, n - 1), rng.randint(1, 6)]; qs.append(q); qs2.append(q)
    b = list(a); exp4 = []; exp5 = []
    for q in qs:
        if q[0] == 1:
            pk = [i for i in range(q[1] + 1, q[2]) if b[i - 1] < b[i] > b[i + 1]]
            exp4.append(pk[q[3] - 1] if q[3] <= len(pk) else -1)
            exp5.append(sum(1 for i in range(q[1] + 2, q[2] - 1) if all(b[i] > b[j] for j in (i - 2, i - 1, i + 1, i + 2))))
        else:
            b[q[1]] = q[2]
    assert kth_peak(a, qs) == exp4
    assert wide_peaks(a, qs2) == exp5
print('Bài 4 (đỉnh thứ k bằng BIT walk) và Bài 5 (đỉnh rộng bán kính 2) khớp brute.')

Bài 4 (đỉnh thứ k bằng BIT walk) và Bài 5 (đỉnh rộng bán kính 2) khớp brute.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
pk[i] = a[i-1] < a[i] > a[i+1]  (0 < i < n-1);  BIT trên pk
[1, l, r] -> prefix(r) - prefix(l+1) nếu r - l >= 2 else 0
[2, x, v] -> a[x] = v; với j ∈ {x-1, x, x+1}: tính lại pk[j], BIT.add(j, mới - cũ)
```

**Checklist:**

- [ ] Chỉ đếm **bên trong** `(l, r)`?
- [ ] Cập nhật tính lại **ba** vị trí `x−1, x, x+1`?
- [ ] Đoạn rỗng (`r − l < 2`) trả 0?
- [ ] So sánh **ngặt** `<`, `>`?
- [ ] Biên `0 < j < n − 1` khi tính lại?